# ConnectorGuard-HDC: replay dataset ke FPGA (Kria KV260)
Dataset sintetis diputar dari DDR (AXI DMA) ke `cg_top` dengan cadence 400 clock/slot (250 kS/s), lalu log per window dibaca kembali dan dibandingkan dengan output RTL di dataset.
Bitstream: `cg_replay.bit` + `cg_replay.hwh` (build dari `rtl/cg_replay_top.sv`, lihat README).

In [ ]:
import time, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pynq import Overlay, allocate
import cg_replay as cg

DATASET = "/home/xilinx/jupyter_notebooks/cg/dataset"   # sesuaikan: folder hasil unzip dataset.zip
BITFILE = "cg_replay.bit"                               # .hwh harus satu folder, nama sama
ol   = Overlay(BITFILE)
dma  = ol.axi_dma_0                 # nama IP mengikuti block design
gpio = ol.axi_gpio_0.mmio           # ch1 out (ctrl) @0x0, ch2 in (stat) @0x8
def ctrl(go=0, rst=0, mirror_en=0, mirror=0):
    gpio.write(0x0, go | rst << 1 | mirror_en << 2 | (mirror & 0x3F) << 8)
def stat():
    s = gpio.read(0x8)
    return dict(ready=s & 1, running=s >> 1 & 1, done=s >> 2 & 1, overflow=s >> 3 & 1,
                last_taken=s >> 4 & 1, window=s >> 6 & 0x3FF, slot=s >> 16)
print(stat())

In [ ]:
stim = cg.pack_stimulus(DATASET)                       # (N, 4) uint32, 1 baris = 1 beat 128 bit
n_valid = int((stim[:, 3] == cg.KIND_SAMPLE).sum())
n_win   = n_valid // 64                                # win_log2 = 6
print("beat:", stim.shape[0], " sampel valid:", n_valid, " window:", n_win)

in_buf  = allocate(shape=stim.shape, dtype=np.uint32);  in_buf[:] = stim;  in_buf.flush()
out_buf = allocate(shape=(n_win + 1, 8), dtype=np.uint32); out_buf[:] = 0   # +1 = penanda akhir

In [ ]:
# reset core, lalu: siapkan penerima log DULU, kirim stimulus, baru go
ctrl(rst=1); time.sleep(0.01); ctrl(rst=0); time.sleep(0.01)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.transfer(in_buf)
t0 = time.time(); ctrl(go=1)
dma.sendchannel.wait(); dma.recvchannel.wait()
print(f"selesai {time.time()-t0:.3f} s   status:", stat())    # replay hardware ~0,18 s; sisanya overhead Python/DMA
out_buf.invalidate()
log = cg.decode_log(np.array(out_buf))
print(len(log), "window")

In [ ]:
res = cg.compare(log, DATASET)       # vs validation/rtl_output.csv (hasil simulasi RTL)
print(res)
assert not res["mismatch"] and res["windows_hw"] == res["windows_ref"], "ada beda hardware vs simulasi"
print("PASS: log hardware sama dengan simulasi RTL pada", res["windows_hw"], "window")

In [ ]:
ref = pd.read_csv(f"{DATASET}/connectorguard_reference.csv")
fig, ax = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
ax[0].step(log.window, log.state, where="post");        ax[0].set_ylabel("state 0..3")
ax[1].plot(log.window, log.health);                     ax[1].set_ylabel("health")
ax[2].plot(log.window, log.r_comp);                     ax[2].set_ylabel("r_comp")
ax[3].step(log.window, log.contactor, where="post", label="contactor")
ax[3].step(log.window, log.hard_trip, where="post", label="hard_trip"); ax[3].legend()
ax[3].set_xlabel("window")
for s, g in ref.groupby("scenario_id"):
    ax[0].axvline(g.window.min(), color="gray", lw=.4); ax[0].text(g.window.min(), 3.2, s, fontsize=7)
plt.tight_layout(); plt.show()

## Tampilan di Pmod (playback lambat)
Replay hardware hanya ~0,18 s, jadi LED tidak bisa mengikuti. Sel ini memutar ulang **log hasil hardware** ke LED dengan kecepatan yang terlihat mata (mode `mirror`).
Ini playback dari log, bukan sinyal langsung dari core. `cp_pwm` ditampilkan statis (menyala bila `cp_duty > 0`).

In [ ]:
SEC_PER_WINDOW = 0.05          # 716 window x 0,05 s = 36 s
start, stop = 1, len(log)      # contoh sempit: kejadian arc S05 -> start, stop = 360, 390
tog = 0
for _, r in log.iloc[start-1:stop].iterrows():
    tog ^= 1
    pins = (int(r.contactor)       | (int(r.state) & 1) << 1 | (int(r.state) >> 1 & 1) << 2 |
            int(r.hard_trip) << 3  | int(r.cp_duty > 0) << 4 | tog << 5)
    ctrl(go=1, mirror_en=1, mirror=pins)
    time.sleep(SEC_PER_WINDOW)
ctrl(go=1, mirror_en=0)        # kembali ke output langsung core